# Rossmann Store Sales — Exploratory Data Analysis

## Objective

Explore the cleaned Rossmann data to understand the main drivers and patterns of store sales before feature engineering and modeling.

The analysis will focus on asking business-oriented questions rather than generating plots without a clear purpose.

## Key Questions

- How are daily sales distributed?
- How strongly are sales related to store opening status?
- How do promotions affect sales?
- How do sales vary across days of the week?
- How do store types and assortment types differ in sales behavior?
- How do holidays affect sales?
- What temporal patterns exist in sales?
- Are there important differences between stores?

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
PROJECT_ROOT = Path.cwd().parent
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

TRAIN_PATH = PROCESSED_DATA_DIR / "train_clean.parquet"
STORE_PATH = PROCESSED_DATA_DIR / "store_clean.parquet"

In [ ]:
TRAIN_PATH.exists(), STORE_PATH.exists()

In [ ]:
train = pd.read_parquet(TRAIN_PATH)
store = pd.read_parquet(STORE_PATH)

In [ ]:
train.shape, store.shape

In [ ]:
train["Open"].value_counts(dropna=False)

In [ ]:
train.groupby("Open")["Sales"].describe()

### Open Status and Sales

Closed-store observations consistently have zero sales, which is logically consistent with the `Open` indicator.

However, some observations with `Open = 1` have zero sales. These cases require further investigation before deciding whether they represent valid business situations or unusual records.

In [ ]:
closed_with_sales = ((train["Open"] == 0) & (train["Sales"] > 0)).sum()

open_with_zero_sales = ((train["Open"] == 1) & (train["Sales"] == 0)).sum()

closed_with_sales, open_with_zero_sales

In [ ]:
train.loc[
    (train["Open"] == 1) & (train["Sales"] == 0),
    ["Store", "Date", "Sales", "Customers", "Promo", "StateHoliday", "SchoolHoliday"],
].head(20)

### Investigation: Open Stores With Zero Sales

There are 54 observations where the store is marked as open but recorded zero sales. These observations are investigated to determine whether they share common characteristics.

In [ ]:
open_zero_sales = train.loc[(train["Open"] == 1) & (train["Sales"] == 0)].copy()

In [ ]:
open_zero_sales.shape

In [ ]:
open_zero_sales["Customers"].value_counts(dropna=False)

In [ ]:
open_zero_sales.loc[
    open_zero_sales["Customers"] > 0,
    ["Store", "Date", "Sales", "Customers", "Promo", "StateHoliday", "SchoolHoliday"],
]

In [ ]:
open_zero_sales["Store"].nunique()

In [ ]:
open_zero_sales["Store"].value_counts().head(10)

In [ ]:
open_zero_sales["Date"].min(), open_zero_sales["Date"].max()

In [ ]:
open_zero_sales["Date"].value_counts().head(10)

### Findings: Open Stores With Zero Sales

- 54 observations have `Open = 1` and `Sales = 0`.
- These observations are distributed across 41 different stores and span from 2013-01-17 to 2015-05-15.
- 52 of the 54 observations also have zero customers.
- Two observations have positive customer counts despite zero recorded sales.
- The pattern is not strongly concentrated in a single store or a single time period.
- These observations are retained for now because there is not enough evidence to classify them as erroneous.

In [ ]:
sales_without_customers = ((train["Sales"] > 0) & (train["Customers"] == 0)).sum()

customers_without_sales = ((train["Sales"] == 0) & (train["Customers"] > 0)).sum()

sales_without_customers, customers_without_sales

### Sales-Customer Consistency

- No observations have positive sales with zero recorded customers.
- Only two observations have positive customer counts with zero sales.
- Overall, `Sales` and `Customers` are highly consistent from a business-logic perspective.
- The two unusual observations are retained because there is insufficient evidence to classify them as erroneous.

In [ ]:
open_days = train.loc[train["Open"] == 1].copy()

In [ ]:
open_days["Promo"].value_counts()

In [ ]:
open_days.groupby("Promo")["Sales"].agg(["count", "mean", "median", "std"])

### Promotion and Sales

Among open-store observations, promotional days are associated with substantially higher sales.

- Mean sales increase from approximately 5,929 without promotion to 8,228 with promotion.
- Median sales increase from 5,459 to 7,649.
- Sales quartiles are consistently higher during promotional periods.
- Sales variability is also higher on promotional days.

These results indicate a strong association between promotion and sales, but they do not establish a causal effect because promotions may coincide with particular stores, weekdays, seasons, or other factors.

In [ ]:
open_days.groupby("Promo")["Customers"].agg(["count", "mean", "median"])

In [ ]:
open_days_with_customers = open_days.loc[open_days["Customers"] > 0].copy()

In [ ]:
open_days_with_customers["SalesPerCustomer"] = (
    open_days_with_customers["Sales"] / open_days_with_customers["Customers"]
)

In [ ]:
open_days_with_customers.groupby("Promo")["SalesPerCustomer"].agg(["mean", "median"])

### Why Are Sales Higher During Promotions?

Promotional days are associated with both:

- Higher customer traffic: mean customer count increases from about 697 to 844.
- Higher sales per customer: mean sales per customer increases from about 8.94 to 10.18.

Therefore, the higher sales observed during promotions appear to be associated with both more customers and greater sales per customer.

This is an association, not yet evidence of a causal effect.

In [ ]:
open_days.groupby("DayOfWeek")["Promo"].agg(["count", "mean"])

In [ ]:
open_days.groupby(["DayOfWeek", "Promo"])["Sales"].agg(["count", "mean", "median"])

### Promotion Effect Across Weekdays

Promotions are only observed on weekdays 1 through 5; no promotional observations occur on days 6 or 7.

Within each weekday from 1 to 5, both mean and median sales are consistently higher on promotional days.

This suggests that the overall association between promotion and higher sales is not explained solely by weekday composition. However, the analysis remains observational and does not establish a causal promotion effect.

### Promotion Association Across Stores

The overall promotion-sales relationship may hide substantial differences between individual stores.

We compare each store's average sales on promotional and non-promotional open days to investigate store-level heterogeneity.

In [ ]:
promo_sales_by_store = (
    open_days.groupby(["Store", "Promo"])["Sales"].mean().unstack("Promo")
)

promo_sales_by_store.head()

In [ ]:
promo_sales_by_store = promo_sales_by_store.rename(
    columns={0: "NoPromoMeanSales", 1: "PromoMeanSales"}
)

promo_sales_by_store.head()

In [ ]:
promo_sales_by_store.isna().sum()

In [ ]:
promo_sales_by_store.shape

In [ ]:
(promo_sales_by_store["NoPromoMeanSales"] == 0).sum()

In [ ]:
comparable_stores = promo_sales_by_store.copy()

comparable_stores["SalesDifference"] = (
    comparable_stores["PromoMeanSales"] - comparable_stores["NoPromoMeanSales"]
)

comparable_stores["SalesUpliftPct"] = (
    comparable_stores["SalesDifference"] / comparable_stores["NoPromoMeanSales"] * 100
)

In [ ]:
comparable_stores["SalesUpliftPct"].describe()

In [ ]:
(comparable_stores["SalesUpliftPct"] > 0).sum()

In [ ]:
(comparable_stores["SalesUpliftPct"] <= 0).sum()

In [ ]:
comparable_stores.nlargest(
    10,
    "SalesUpliftPct",
)

In [ ]:
comparable_stores.nsmallest(
    10,
    "SalesUpliftPct",
)

### Store-Level Promotion Findings

- All 1,115 stores have observations for both promotional and non-promotional open days.
- Promotional days are associated with higher average sales in 1,114 of the 1,115 stores.
- The median store-level sales uplift is approximately 40.6%.
- Promotion-associated uplift varies substantially across stores, indicating strong store-level heterogeneity.
- One store, Store 274, shows lower average sales during promotional days.
- These comparisons are observational and may still be influenced by seasonality, weekday patterns, or promotion timing.

In [ ]:
open_days.loc[open_days["Store"] == 274].groupby("Promo")["Sales"].agg(
    ["count", "mean", "median", "std"]
)

In [ ]:
open_days.loc[open_days["Store"] == 274].groupby(["DayOfWeek", "Promo"])["Sales"].agg(
    ["count", "mean", "median"]
)

In [ ]:
open_days.loc[open_days["Store"] == 198].groupby("Promo")["Sales"].agg(
    ["count", "mean", "median", "std"]
)

### Investigation of Store 274

Store 274 initially appeared to have lower average sales during promotions.

However, weekday-level analysis shows that promotional sales are higher on days 2 through 5 and nearly unchanged on day 1.

The negative overall uplift is partly explained by weekday composition: days 6 and 7 never contain promotions, and Store 274 has particularly high non-promotional sales on day 7.

Therefore, the raw store-level comparison is confounded by day-of-week effects.

In [ ]:
weekday_open_days = open_days.loc[open_days["DayOfWeek"].between(1, 5)].copy()

In [ ]:
weekday_promo_by_store = (
    weekday_open_days.groupby(["Store", "Promo"])["Sales"]
    .mean()
    .unstack("Promo")
    .rename(
        columns={
            0: "NoPromoMeanSales",
            1: "PromoMeanSales",
        }
    )
)

In [ ]:
weekday_promo_by_store["SalesUpliftPct"] = (
    (
        weekday_promo_by_store["PromoMeanSales"]
        - weekday_promo_by_store["NoPromoMeanSales"]
    )
    / weekday_promo_by_store["NoPromoMeanSales"]
    * 100
)

In [ ]:
weekday_promo_by_store["SalesUpliftPct"].describe()

In [ ]:
weekday_promo_by_store.loc[274]

In [ ]:
(weekday_promo_by_store["SalesUpliftPct"] <= 0).sum()

### Weekday-Adjusted Store-Level Promotion Findings

After restricting the comparison to weekdays 1 through 5, all 1,115 stores show higher average sales on promotional days.

- Mean store-level uplift: approximately 40.0%.
- Median store-level uplift: approximately 39.5%.
- Minimum uplift: approximately 1.9%.
- Maximum uplift: approximately 102.1%.
- No store shows non-positive uplift after removing weekend composition effects.

Store 274 changes from a negative raw uplift to approximately +8.7%, confirming that its initial result was confounded by weekday composition.

The promotion-sales relationship remains observational and should not be interpreted as causal.

In [ ]:
eda = train.merge(store, on="Store", how="left", validate="many_to_one")

In [ ]:
eda.shape

In [ ]:
eda["StoreType"].isna().sum()

In [ ]:
open_eda = eda.loc[eda["Open"] == 1].copy()

open_eda.groupby("StoreType")["Sales"].agg(["count", "mean", "median", "std"])

### Sales by Store Type

Among open-store observations, StoreType `b` shows substantially higher mean and median sales than the other store types.

StoreTypes `a`, `c`, and `d` have relatively similar average sales levels, while type `b` also shows greater sales variability.

These differences are descriptive and should not be interpreted as a causal effect of store type.

In [ ]:
open_eda.groupby("StoreType")["Customers"].agg(["count", "mean", "median"])

In [ ]:
open_eda_with_customers = open_eda.loc[open_eda["Customers"] > 0].copy()

open_eda_with_customers["SalesPerCustomer"] = (
    open_eda_with_customers["Sales"] / open_eda_with_customers["Customers"]
)

In [ ]:
open_eda_with_customers.groupby("StoreType")["SalesPerCustomer"].agg(["mean", "median"])

### Customer Behavior by Store Type

StoreType `b` has substantially higher daily customer traffic than the other store types, which explains much of its higher total sales.

However, StoreType `b` has the lowest sales per customer, while StoreType `d` has the highest sales per customer despite having the lowest average customer traffic.

This suggests that store types differ not only in total sales, but also in the way those sales are generated:
- Type `b`: high traffic, lower sales per customer.
- Type `d`: lower traffic, higher sales per customer.
- Types `a` and `c`: relatively similar customer and spending patterns.

In [ ]:
store["StoreType"].value_counts()

In [ ]:
pd.crosstab(
    store["StoreType"],
    store["Assortment"],
    normalize="index",
) * 100

### Store Type and Assortment Relationship

Store type and assortment are not distributed independently.

- StoreType `a` is mostly associated with Assortment `a`.
- StoreType `d` is mostly associated with Assortment `c`.
- Assortment `b` appears only within StoreType `b`.
- StoreType `b` contains only 17 stores, so comparisons involving this group should be interpreted cautiously.

This relationship means that differences in sales across store types may partly reflect differences in assortment composition.

In [ ]:
open_eda.groupby("Assortment")["Sales"].agg(["count", "mean", "median", "std"])

In [ ]:
open_eda.groupby(["StoreType", "Assortment"])["Sales"].agg(["count", "mean", "median"])

### Sales by Store Type and Assortment

Aggregate sales differ across assortment categories, but assortment composition is strongly related to store type.

Within StoreTypes `a` and `d`, Assortment `c` is associated with higher average sales than Assortment `a`. The difference is smaller within StoreType `c`.

StoreType `b` shows large differences across assortment categories, but these results should be interpreted cautiously because some StoreType-Assortment combinations contain very few stores.

Therefore, aggregate assortment comparisons may be confounded by store type.

In [ ]:
store_level_sales = (
    open_eda.groupby(["Store", "StoreType", "Assortment"])["Sales"]
    .mean()
    .reset_index(name="MeanSales")
)

store_level_sales.head()

In [ ]:
store_level_sales.groupby(["StoreType", "Assortment"])["MeanSales"].agg(
    ["count", "mean", "median"]
)

## Temporal Sales Patterns

### Sales by Day of Week

Analyze whether sales and customer traffic differ across days of the week.

In [ ]:
sales_by_day = open_eda.groupby("DayOfWeek").agg(
    observation_count=("Sales", "size"),
    mean_sales=("Sales", "mean"),
    median_sales=("Sales", "median"),
    mean_customers=("Customers", "mean"),
    median_customers=("Customers", "median"),
)

sales_by_day

### Findings

Sales and customer traffic vary noticeably across days of the week.

Monday shows relatively high sales and customer traffic, while Saturday has the lowest average sales among open stores.

Sunday shows very high average sales and customer traffic, but the number of open-store observations is much smaller than on other days. Therefore, Sunday results should be interpreted cautiously, as only a small subset of stores appears to operate on Sundays.

In [ ]:
open_rate_day = eda.groupby("DayOfWeek")["Open"].agg(
    observation_counts="size", open_rate="mean"
)
open_rate_day

### Finding

Store opening patterns differ substantially across days of the week.

Most stores are open on weekdays and Saturdays, while only about 2.5% of Sunday observations correspond to open stores. Therefore, the high average Sunday sales observed earlier represent a small and potentially distinct subset of stores and should not be generalized to all stores.

### Sales Trend Over Time

Analyze how sales change over time and identify possible trends, seasonal patterns, or unusual periods.

In [ ]:
daily_sales = (
    open_eda.groupby("Date")
    .agg(
        mean_sales=("Sales", "mean"),
        median_sales=("Sales", "median"),
        total_sales=("Sales", "sum"),
        open_store_count=("Sales", "nunique"),
    )
    .reset_index()
)

daily_sales.head()

### Daily Average Sales Trend

Visualize the daily average sales of open stores to examine changes and recurring patterns over time.

In [ ]:
plt.figure(figsize=(14, 5))

plt.plot(
    daily_sales["Date"],
    daily_sales["mean_sales"],
)

plt.title("Daily Average Sales Over Time")
plt.xlabel("Date")
plt.ylabel("Average Sales")
plt.show()

### Rolling Average of Daily Sales

Use rolling averages to smooth short-term daily fluctuations and make the underlying sales pattern easier to observe.

In [ ]:
daily_sales["mean_sales_7d"] = daily_sales["mean_sales"].rolling(window=7).mean()

daily_sales[["Date", "mean_sales", "mean_sales_7d"]].head(10)

In [ ]:
plt.figure(figsize=(14, 5))

plt.plot(
    daily_sales["Date"],
    daily_sales["mean_sales"],
    label="Daily Mean Sales",
    alpha=0.4,
)

plt.plot(
    daily_sales["Date"],
    daily_sales["mean_sales_7d"],
    label="7-Day Rolling Average",
)

plt.title("Daily Average Sales and 7-Day Rolling Average")
plt.xlabel("Date")
plt.ylabel("Average Sales")
plt.legend()

plt.show()

### Finding

Daily average sales show substantial short-term fluctuations.

The 7-day rolling average smooths some of this daily variation and reveals periods of higher and lower sales, including noticeable increases around the end of 2013 and 2014. These patterns suggest that sales have important time-related structure that should be investigated further.

In [ ]:
daily_sales["mean_sales_30d"] = daily_sales["mean_sales"].rolling(window=30).mean()

In [ ]:
plt.figure(figsize=(14, 5))

plt.plot(
    daily_sales["Date"],
    daily_sales["mean_sales_7d"],
    label="7-Day Rolling Average",
)

plt.plot(
    daily_sales["Date"],
    daily_sales["mean_sales_30d"],
    label="30-Day Rolling Average",
)

plt.title("7-Day vs 30-Day Rolling Average Sales")
plt.xlabel("Date")
plt.ylabel("Average Sales")
plt.legend()

plt.show()

### Finding

The 30-day rolling average provides a clearer view of the medium-term sales pattern than the 7-day rolling average.

Several periods of elevated sales are visible, particularly near the end of 2013 and 2014. However, the causes of these increases cannot yet be determined and require further investigation using promotions, holidays, and seasonal variables.

In [ ]:
monthly_sales = (
    daily_sales.set_index("Date")["mean_sales"]
    .resample("MS")
    .mean()
    .reset_index(name="mean_sales")
)
monthly_sales.head()

In [ ]:
plt.figure(figsize=(14, 5))

plt.plot(
    monthly_sales["Date"],
    monthly_sales["mean_sales"],
    marker="o",
)

plt.title("Monthly Average Sales Over Time")
plt.xlabel("Date")
plt.ylabel("Average Sales")
plt.grid(alpha=0.3)

plt.show()

### Monthly Sales Comparison Across Years

Compare the same calendar months across different years to investigate recurring seasonal sales patterns.

In [ ]:
monthly_sales["Year"] = monthly_sales["Date"].dt.year
monthly_sales["Month"] = monthly_sales["Date"].dt.month

monthly_comparison = monthly_sales.pivot(
    index="Month",
    columns="Year",
    values="mean_sales",
)

monthly_comparison.round(0)

### Finding

December has the highest average sales in both 2013 and 2014, suggesting a recurring year-end sales pattern.

However, other months show less consistent patterns across years. Further analysis is needed to investigate whether promotions or holidays contribute to the December peaks.

In [ ]:
promo_by_month = (
    open_eda.assign(Year=open_eda["Date"].dt.year, Month=open_eda["Date"].dt.month)
    .groupby(["Month", "Year"])["Promo"]
    .mean()
    .unstack("Year")
    .mul(100)
    .round(1)
)

promo_by_month

### Finding

December shows the highest average sales in both 2013 and 2014, despite not having the highest promotion rate.

Therefore, the December sales peaks cannot be explained by promotion frequency alone. Other factors, such as holidays and seasonal shopping behavior, should be investigated.

In [ ]:
pd.crosstab(
    eda["StateHoliday"],
    eda["Open"],
    normalize="index",
) * 100

### Finding

Most stores are closed on state holidays, with opening rates below 4% across all holiday categories.

Christmas holidays have the lowest opening rate (1.73%). This suggests that the high December sales may be related to increased shopping before Christmas rather than sales on the holidays themselves.

In [ ]:
december_sales = daily_sales.loc[
    (daily_sales["Date"].dt.month == 12) & (daily_sales["Date"].dt.day >= 15),
    ["Date", "mean_sales", "open_store_count"],
].copy()

december_sales

### Finding: Sales Around Christmas

Sales are noticeably higher before Christmas, particularly around December 23, in both 2013 and 2014.

Average sales drop sharply on December 24 and December 31.

Although average sales appear high on December 25 and 26, very few stores are open on these dates. Therefore, these averages are not representative of the full store population.

These observations support a possible pre-Christmas shopping pattern, but further investigation is needed.

In [ ]:
target_dates = pd.to_datetime(["2014-12-23", "2014-12-25"])

same_stores = (
    open_eda.loc[
        open_eda["Date"].isin(target_dates),
        ["Store", "Date", "Sales"],
    ]
    .pivot(index="Store", columns="Date", values="Sales")
    .dropna()
)

same_stores

In [ ]:
same_stores["sales_difference"] = (
    same_stores[pd.Timestamp("2014-12-23")] - same_stores[pd.Timestamp("2014-12-25")]
)

same_stores.head()

In [ ]:
print("Number of stores:", len(same_stores))

print(
    "Mean sales on Dec 23:",
    same_stores[pd.Timestamp("2014-12-23")].mean(),
)

print(
    "Mean sales on Dec 25:",
    same_stores[pd.Timestamp("2014-12-25")].mean(),
)

print(
    "Average sales difference:",
    same_stores["sales_difference"].mean(),
)

In [ ]:
differences = same_stores["sales_difference"]

print("Higher sales on Dec 23:", (differences > 0).sum())
print("Higher sales on Dec 25:", (differences < 0).sum())
print("Equal sales:", (differences == 0).sum())

print("Mean difference:", differences.mean())
print("Median difference:", differences.median())

### Finding: Same-Store Christmas Comparison

Among the 16 stores open on both December 23 and 25, 2014, average sales were higher on December 23 by approximately 3,268 units.

This comparison controls for differences in store identity, but individual stores show varying sales patterns. Further analysis of the median difference and the number of stores with positive differences is needed.

In [ ]:
open_eda.groupby("SchoolHoliday")["Sales"].agg(["count", "mean", "median"])

### Finding: School Holidays and Sales

Open stores have approximately 4.4% higher average sales during school holidays.

Median sales are also higher during school holidays.

However, this relationship does not establish causation. Other factors, such as promotions and seasonal patterns, may contribute to the difference.

In [ ]:
school_holiday_promo = open_eda.groupby("SchoolHoliday")["Promo"].agg(
    observation_count="size",
    promo_rate="mean",
)

school_holiday_promo["promo_rate_pct"] = school_holiday_promo["promo_rate"] * 100

school_holiday_promo

In [ ]:
school_holiday_sales_by_promo = open_eda.groupby(["SchoolHoliday", "Promo"])[
    "Sales"
].agg(["count", "mean", "median"])

school_holiday_sales_by_promo

### Finding: School Holidays and Promotions

Average sales are higher during school holidays in both promotional and non-promotional periods.

The difference is approximately 4.9% without promotions and 1.7% with promotions.

Therefore, the higher sales observed during school holidays cannot be explained by promotion frequency alone. Seasonal and store-level differences may also contribute.

In [ ]:
school_holiday_by_month = (
    open_eda.assign(Month=open_eda["Date"].dt.month)
    .groupby("Month")["SchoolHoliday"]
    .mean()
    .mul(100)
    .round(1)
)

school_holiday_by_month

### Finding: Monthly Distribution of School Holidays

School holidays are not evenly distributed throughout the year.

They are particularly common in July (44.4%) and August (62.5%), while November has very few school-holiday observations (0.4%).

Therefore, month of the year may influence the observed relationship between school holidays and sales.

In [ ]:
monthly_school_sales = (
    open_eda.assign(Month=open_eda["Date"].dt.month)
    .groupby(["Month", "SchoolHoliday"])["Sales"]
    .agg(["count", "mean", "median"])
)

monthly_school_sales

### Finding: School Holidays and Monthly Sales

Average sales during school holidays are higher in 10 out of 12 calendar months.

September and December are exceptions, with lower average sales during school holidays.

This suggests that the relationship between school holidays and sales varies by month. However, promotions, store differences, and other temporal factors may still influence these results.

In [ ]:
school_holiday_controlled = (
    open_eda.assign(Month=open_eda["Date"].dt.month)
    .groupby(["Month", "Promo", "SchoolHoliday"])["Sales"]
    .agg(["count", "mean", "median"])
)

school_holiday_controlled

### Finding: School Holidays, Promotions, and Seasonality

School holidays are associated with higher average sales overall.

When comparing sales within the same calendar month and promotion status, the relationship varies across groups.

For example, school-holiday sales are higher in January and July under both promotion conditions, while the opposite pattern appears in August.

These findings suggest that the relationship between school holidays and sales depends on other factors and cannot be explained by promotion frequency or month alone.

Further investigation may be useful during modeling.

In [ ]:
competition_sales = store_level_sales.merge(
    store[["Store", "CompetitionDistance"]],
    on="Store",
    how="left",
    validate="one_to_one",
)

competition_sales[["CompetitionDistance", "MeanSales"]].describe()

In [ ]:
competition_valid = competition_sales.dropna(subset=["CompetitionDistance"])

plt.figure(figsize=(10, 6))

plt.scatter(
    competition_valid["CompetitionDistance"],
    competition_valid["MeanSales"],
    alpha=0.4,
    s=15,
)

plt.xscale("log")

plt.title("Competition Distance vs Average Store Sales")
plt.xlabel("Competition Distance (meters, log scale)")
plt.ylabel("Average Daily Sales")
plt.show()

In [ ]:
spearman_corr = competition_valid["CompetitionDistance"].corr(
    competition_valid["MeanSales"],
    method="spearman",
)

print("Spearman correlation:", spearman_corr)

### Finding: Competition Distance and Sales

The Spearman correlation between competition distance and average store sales is approximately -0.028, indicating a negligible overall monotonic relationship.

This result is consistent with the scatter plot, which shows no clear increasing or decreasing sales pattern as competition distance changes.

However, competition distance may still be useful when combined with other store characteristics.

In [ ]:
for store_type, group in competition_valid.groupby("StoreType"):
    correlation = group["CompetitionDistance"].corr(
        group["MeanSales"],
        method="spearman",
    )

    print(f"StoreType {store_type}: n={len(group)}, Spearman={correlation:.3f}")

### Finding: Competition Distance by Store Type

The overall Spearman correlation between competition distance and average store sales is approximately -0.028, indicating a negligible monotonic relationship.

Store-level correlations are also weak for StoreTypes `a`, `c`, and `d`.

StoreType `b` shows a more negative correlation (-0.319), but this result should be interpreted cautiously because the group contains only 17 stores.

Overall, competition distance alone does not show a strong relationship with sales, although it may still provide useful information when combined with other features.

### Promo2 and Store Sales

Compare average store sales between stores participating and not participating in the Promo2 program.

In [ ]:
promo2_sales = store_level_sales.merge(
    store[["Store", "Promo2"]],
    on="Store",
    how="left",
    validate="one_to_one",
)

promo2_sales.groupby("Promo2")["MeanSales"].agg(["count", "mean", "median"])

### Finding: Promo2 and Store Sales

Stores not participating in Promo2 have higher average daily sales (7,337) than participating stores (6,550).

Median sales show a similar pattern.

However, this difference does not imply that Promo2 reduces sales. Differences in store characteristics and the actual promotion periods must be investigated before drawing further conclusions.

In [ ]:
pd.crosstab(
    promo2_sales["StoreType"],
    promo2_sales["Promo2"],
)

In [ ]:
promo2_sales_by_type = promo2_sales.groupby(["StoreType", "Promo2"])["MeanSales"].agg(
    ["count", "mean", "median"]
)

promo2_sales_by_type

### Finding: Promo2 Sales by Store Type

Stores without Promo2 have higher average sales across all four store types.

This suggests that differences in store type composition alone do not explain the overall sales gap.

However, Promo2 membership does not indicate that the program was active throughout the entire observation period. The program's starting date and promotion schedule must also be considered.

Therefore, no causal conclusion can be drawn from these comparisons.

In [ ]:
promo2_start_years = (
    store.loc[
        store["Promo2"] == 1,
        "Promo2SinceYear",
    ]
    .value_counts()
    .sort_index()
)

promo2_start_years

### Finding: Promo2 Start Years

Among 571 Promo2 stores, 346 started the program before 2013, while 225 have start years between 2013 and 2015.

Since the sales dataset covers January 2013 to July 2015, Promo2 membership does not necessarily indicate that the program had started on every recorded date.

Both the start week/year and promotional months must be considered when analyzing Promo2 activity.

In [ ]:
promo2_intervals = store.loc[
    store["Promo2"] == 1,
    "PromoInterval",
].value_counts(dropna=False)

promo2_intervals

### Finding: Promo2 Intervals

Promo2 stores follow three different quarterly promotion schedules.

The most common schedule is January, April, July, and October, used by 335 stores.

The other schedules are used by 130 and 106 stores, respectively.

PromoInterval identifies scheduled promotion months, but the program's starting date must also be considered when analyzing historical sales.

In [ ]:
promo2_daily = open_eda.loc[
    open_eda["Promo2"] == 1,
    [
        "Store",
        "Date",
        "Sales",
        "Promo2SinceYear",
        "Promo2SinceWeek",
        "PromoInterval",
    ],
].copy()

promo2_daily["Month"] = promo2_daily["Date"].dt.month

promo2_daily.head()

In [ ]:
start_week = (
    promo2_daily["Promo2SinceYear"].astype(str)
    + "-W"
    + promo2_daily["Promo2SinceWeek"].astype(str).str.zfill(2)
    + "-1"
)

promo2_daily["Promo2StartDate"] = pd.to_datetime(
    start_week,
    format="%G-W%V-%u",
)

promo2_daily[
    [
        "Store",
        "Date",
        "Promo2SinceYear",
        "Promo2SinceWeek",
        "Promo2StartDate",
    ]
].head()

In [ ]:
promo2_daily["HasPromo2Started"] = (
    promo2_daily["Date"] >= promo2_daily["Promo2StartDate"]
)

promo2_daily["HasPromo2Started"].value_counts()

In [ ]:
interval_months = {
    "Jan,Apr,Jul,Oct": [1, 4, 7, 10],
    "Feb,May,Aug,Nov": [2, 5, 8, 11],
    "Mar,Jun,Sept,Dec": [3, 6, 9, 12],
}

promo2_daily["IsPromo2Month"] = [
    month in interval_months[interval]
    for month, interval in zip(
        promo2_daily["Month"], promo2_daily["PromoInterval"], strict=True
    )
]

promo2_daily["IsPromo2Month"].value_counts()

In [ ]:
promo2_daily["IsPromo2Scheduled"] = (
    promo2_daily["HasPromo2Started"] & promo2_daily["IsPromo2Month"]
)

promo2_daily["IsPromo2Scheduled"].value_counts()

In [ ]:
post_start_promo2 = promo2_daily.loc[promo2_daily["HasPromo2Started"]].copy()

In [ ]:
store_schedule_sales = (
    post_start_promo2.groupby(["Store", "IsPromo2Scheduled"])["Sales"]
    .mean()
    .unstack("IsPromo2Scheduled")
    .rename(
        columns={
            False: "OffScheduleMeanSales",
            True: "ScheduledMeanSales",
        }
    )
)

store_schedule_sales.head()

In [ ]:
print("Total stores:", len(store_schedule_sales))

print(
    "Stores with both groups:",
    len(store_schedule_sales.dropna()),
)

In [ ]:
store_schedule_sales["SalesDifference"] = (
    store_schedule_sales["ScheduledMeanSales"]
    - store_schedule_sales["OffScheduleMeanSales"]
)

store_schedule_sales["SalesDifferencePct"] = (
    store_schedule_sales["SalesDifference"]
    / store_schedule_sales["OffScheduleMeanSales"]
    * 100
)

In [ ]:
print(
    "Higher sales during scheduled months:",
    (store_schedule_sales["SalesDifference"] > 0).sum(),
)

print(
    "Lower sales during scheduled months:",
    (store_schedule_sales["SalesDifference"] < 0).sum(),
)

print(
    "Equal sales:",
    (store_schedule_sales["SalesDifference"] == 0).sum(),
)

store_schedule_sales["SalesDifferencePct"].describe()

### Finding: Promo2 Scheduled vs. Off-Schedule Sales

Among 571 Promo2 stores, 143 have higher average sales during scheduled months, while 428 have lower average sales.

The mean percentage difference is -1.67%, and the median difference is -2.55%.

Although scheduled months are associated with lower sales for most stores, this does not establish a negative effect of Promo2. Seasonal patterns and regular promotions may influence the comparison.

In [ ]:
promo2_with_promo = promo2_daily.merge(
    open_eda[["Store", "Date", "Promo"]],
    on=["Store", "Date"],
    how="left",
    validate="one_to_one",
)

In [ ]:
promo2_with_promo.loc[promo2_with_promo["HasPromo2Started"]].groupby(
    "IsPromo2Scheduled"
)["Promo"].agg(["count", "mean"])

### Promo2 Analysis — Key Findings

Stores without Promo2 have higher average sales than participating stores.

Among 571 Promo2 stores, 428 show lower average sales during scheduled months, with a mean difference of -1.67%.

Regular promotion rates are similar between scheduled (45.24%) and off-schedule (44.56%) months.

These results do not establish that Promo2 reduces sales. Seasonal effects, store characteristics, and other factors may influence the observed differences.

The derived `IsPromo2Scheduled` feature may be useful for future modeling, but it represents scheduled periods rather than confirmed promotion activity.

## Sales Distribution and Outlier Analysis

Examine the distribution of daily sales, identify extreme values, and assess whether they represent potentially meaningful business observations.

In [ ]:
sales_distribution = open_eda["Sales"].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
)

sales_distribution

In [ ]:
plt.figure(figsize=(12, 5))

plt.hist(
    open_eda["Sales"],
    bins=60,
    edgecolor="white",
)

plt.title("Distribution of Daily Sales")
plt.xlabel("Daily Sales")
plt.ylabel("Number of Observations")

plt.show()

### Finding: Sales Distribution

Daily sales are right-skewed, with most observations concentrated between approximately 3,000 and 10,000.

A long right tail indicates that some store-day observations have unusually high sales.

These extreme values should be investigated rather than automatically removed, as they may reflect genuine business activity.

In [ ]:
q1 = open_eda["Sales"].quantile(0.25)
q3 = open_eda["Sales"].quantile(0.75)

iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

sales_outliers = open_eda.loc[
    (open_eda["Sales"] < lower_bound) | (open_eda["Sales"] > upper_bound)
].copy()

print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print("Number of outliers:", len(sales_outliers))
print(
    "Outlier percentage:",
    len(sales_outliers) / len(open_eda) * 100,
)

In [ ]:
outlier_by_store_type = (
    open_eda.assign(IsSalesOutlier=open_eda["Sales"] > upper_bound)
    .groupby("StoreType")
    .agg(
        observation_count=("Sales", "size"),
        outlier_count=("IsSalesOutlier", "sum"),
        outlier_rate=("IsSalesOutlier", "mean"),
    )
)

outlier_by_store_type["outlier_rate_pct"] = outlier_by_store_type["outlier_rate"] * 100

outlier_by_store_type

### Finding: Outliers by Store Type

StoreType `b` has the highest proportion of sales observations above the global IQR threshold (23.1%).

StoreType `a` has the largest absolute number of outliers, partly because it contains many more observations.

Since store types have different typical sales levels, applying a single global outlier threshold may classify normal high sales as outliers.

Store-type-specific sales distributions should be examined before making decisions about outlier treatment.

In [ ]:
sales_percentiles_by_type = (
    open_eda.groupby("StoreType")["Sales"].quantile([0.50, 0.95, 0.99]).unstack()
)

sales_percentiles_by_type

### Finding: Store-Type Sales Percentiles

Sales distributions differ substantially across store types.

StoreType `b` has the highest median (9,130), 95th percentile (19,842), and 99th percentile (26,116).

This explains why applying a single global IQR threshold identifies a particularly high proportion of observations from StoreType `b` as outliers.

Outlier detection should consider differences between store types and individual stores. Extreme sales values should not be automatically removed.

In [ ]:
high_sales_by_month = (
    open_eda.assign(
        Month=open_eda["Date"].dt.month,
        IsHighSales=open_eda["Sales"] > upper_bound,
    )
    .groupby("Month")
    .agg(
        observation_count=("Sales", "size"),
        high_sales_count=("IsHighSales", "sum"),
        high_sales_rate=("IsHighSales", "mean"),
    )
)

high_sales_by_month["high_sales_pct"] = high_sales_by_month["high_sales_rate"] * 100

high_sales_by_month

### Sales Outlier Analysis — Key Findings

Daily sales are right-skewed, with most observations concentrated between approximately 3,000 and 10,000.

Using the global 1.5 × IQR rule, 30,769 observations (3.64%) are identified as statistical outliers.

Outlier rates differ across store types. StoreType `b` has the highest rate (23.1%), reflecting its generally higher sales distribution.

December has the highest proportion of high-sales observations (10.58%), supporting the presence of a recurring year-end sales pattern.

**Decision:** High-sales observations will be retained at this stage, since they may reflect genuine store characteristics and seasonal shopping behavior rather than data errors.

## EDA Summary and Modeling Implications

### 1. Key Findings

**Store Operations and Sales**
- Closed stores consistently report zero sales.
- A small number of open-store observations have zero sales, including two cases with positive customer counts.
- These unusual observations were retained because there is insufficient evidence that they are data errors.

**Promotions and Customer Behavior**
- Regular promotions are associated with higher sales and customer traffic.
- The positive relationship remains visible when comparing the same stores and weekdays.
- These associations do not establish a causal effect of promotions.

**Temporal Patterns and Holidays**
- Sales and store opening rates vary across days of the week.
- December has the highest monthly average sales in both 2013 and 2014.
- School holidays are associated with slightly higher sales overall, but the relationship varies across months and promotion conditions.

**Store Characteristics**
- Sales patterns differ across StoreType and Assortment categories.
- StoreType `b` has higher average sales and customer traffic, but contains only 17 stores.
- StoreType and Assortment are related, so their effects should not be interpreted independently without further analysis.

**Competition and Promo2**
- CompetitionDistance has a negligible overall Spearman correlation with average store sales (approximately -0.028).
- Promo2 participation and scheduled months show sales differences, but these comparisons do not establish the effectiveness of Promo2.
- Promo2 start dates and scheduled months may provide useful information for future prediction.

**Sales Distribution and Outliers**
- Daily sales are right-skewed.
- The global IQR method identifies 30,769 observations (3.64%) as statistical outliers.
- High-sales observations are particularly common in December and StoreType `b`.
- Extreme sales observations will be retained because they may represent genuine business behavior.

### 2. Modeling Implications

**Target Variable**

The prediction target is `Sales` for each Store-Date combination.

**Potential Features**

- Store characteristics: Store, StoreType, Assortment
- Calendar features: DayOfWeek, Month, and other date-derived features
- Business features: Promo, StateHoliday, SchoolHoliday
- Competition features: CompetitionDistance and available competition timing information
- Promo2 features: participation, start date, and scheduled promotional months

Feature availability at prediction time must be verified.

**Data Leakage Prevention**

`Customers` was useful for exploratory analysis but should not be used as a predictor unless its value is genuinely available when making forecasts.

Any historical sales-based features must use only information available before the prediction date.

**Handling Missing Values and Outliers**

- Preserve legitimate high-sales observations.
- Treat structural missing values separately from unknown measurements.
- Avoid making unsupported assumptions when filling missing competition-related information.

**Validation Strategy**

Use chronological Train/Validation splits instead of random row splitting.

Fit preprocessing steps and calculate historical statistics using training data only.

Consider the known gaps in historical coverage for some stores when interpreting validation results.

**Baseline Modeling**

Start with a simple, reproducible baseline using historical training data before experimenting with more complex models.

Evaluate predictions using appropriate regression metrics, such as MAE and RMSE, and compare future models against the baseline.

### 3. Final Conclusion

The EDA reveals important temporal, promotional, and store-specific sales patterns.

However, the observed relationships are descriptive and do not prove causation.

The next stage is to build a leakage-free baseline forecasting model and assess whether the identified features improve predictive performance.